## Phase 4 - Deep Learning Models

In [2]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
    Dropout,
    Input
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.sequence import pad_sequences


In [3]:
data = pd.read_csv("imdb_cleaned.csv")
print(data.shape)

(49582, 4)


In [4]:

X = data["clean_review"]
y = data["label"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print(len(X_train))
print(len(X_val))
print(len(X_test))

39665
4958
4959


In [6]:
import pickle

with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

print(len(tokenizer.word_index))

90662


In [7]:
MAX_SEQUENCE_LENGTH = 500

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_integer = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_integer = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_integer = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train = np.asarray(y_train)
y_val = np.asarray(y_val)
y_test = np.asarray(y_test)

print(X_train_integer.shape)
print(X_val_integer.shape)
print(X_test_integer.shape)

(39665, 500)
(4958, 500)
(4959, 500)


In [8]:
def evaluate_model(model, X_test, y_test, model_name, batch_size=64):

    # Generate probabilities
    probabilities = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0
    ).ravel()

    # Convert probabilities to class predictions
    predictions = (probabilities >= 0.5).astype(int)

    # metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test,  predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    # Classification Report
    report = classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"],
        digits=4
    )

    # Print results
    print("=" * 60)
    print(f"{model_name} RESULTS")
    print("=" * 60)

    print(f"Accuracy : {accuracy:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall   : {recall:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print(f"ROC-AUC  : {roc_auc:.5f}")

    print("\nClassification Report")
    print("-" * 60)
    print(report)

    print("Confusion Matrix")
    print(cm)

    # Return everything for later comparison
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    return results, probabilities, predictions, cm

### 1. ANN

In [9]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
MAX_SEQUENCE_LENGTH = 500

BATCH_SIZE = 64
EPOCHS = 10


In [37]:
ann_model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    ),

    GlobalAveragePooling1D(),
    Dense(128, activation="relu"),

    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [43]:
print("X_train:", X_train_integer.shape)
print("X_val  :", X_val_integer.shape)
print("X_test :", X_test_integer.shape)

X_train: (39665, 500)
X_val  : (4958, 500)
X_test : (4959, 500)


In [38]:
start_time = time.time()

ann_history = ann_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

ann_training_time = time.time() - start_time

print(f"\nANN training time: {ann_training_time:.2f} seconds")

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.7092 - loss: 0.5310 - val_accuracy: 0.7729 - val_loss: 0.4674
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8581 - loss: 0.3290 - val_accuracy: 0.8875 - val_loss: 0.2794
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8865 - loss: 0.2720 - val_accuracy: 0.8935 - val_loss: 0.2641
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9000 - loss: 0.2451 - val_accuracy: 0.8945 - val_loss: 0.2598
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9121 - loss: 0.2204 - val_accuracy: 0.8697 - val_loss: 0.3142
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9213 - loss: 0.1979 - val_accuracy: 0.8919 - val_loss: 0.2680
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9202 - loss: 0.1979 - val_accuracy: 0.8917 - val_loss: 0.2744
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9332 - loss: 0.1687 - val_accuracy: 0.

In [39]:
ann_results, ann_probabilities, ann_predictions, ann_cm = evaluate_model(
    model=ann_model,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="ANN Baseline",
    batch_size=64
)

ANN Baseline RESULTS
Accuracy : 0.75015
Precision: 0.96712
Recall   : 0.51989
F1 Score : 0.67625
ROC-AUC  : 0.95225

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.6700    0.9822    0.7966      2470
    Positive     0.9671    0.5199    0.6762      2489

    accuracy                         0.7502      4959
   macro avg     0.8185    0.7510    0.7364      4959
weighted avg     0.8191    0.7502    0.7362      4959

Confusion Matrix
[[2426   44]
 [1195 1294]]


In [40]:
ann_model.save("ann_baseline.keras")
print("saved")

saved


In [41]:
import pickle

with open("ann_baseline_history.pkl", "wb") as file:
    pickle.dump(ann_history.history, file)

print("history saved")

history saved


In [51]:
baseline_results = []
baseline_results.append(ann_results)

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750151,0.967115,0.519888,0.676248,0.952254


### 2. Simple RNN

In [31]:
rnn_model = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    SimpleRNN(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,881,217 (14.81 MB)

 Trainable params: 3,881,217 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
rnn_history = rnn_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)


Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 45ms/step - accuracy: 0.4983 - loss: 0.7034 - val_accuracy: 0.5052 - val_loss: 0.6980
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5038 - loss: 0.6994 - val_accuracy: 0.4994 - val_loss: 0.6994
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5065 - loss: 0.6968 - val_accuracy: 0.4968 - val_loss: 0.6978
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5057 - loss: 0.6958 - val_accuracy: 0.5030 - val_loss: 0.6936
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5082 - loss: 0.6932 - val_accuracy: 0.5071 - val_loss: 0.6930
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5160 - loss: 0.6894 - val_accuracy: 0.5050 - val_loss: 0.6970
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5239 - loss: 0.6829 - val_accuracy: 0.5006 - val_loss: 0.6978
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 43ms/step - accuracy: 0.5190 - loss: 0.6783 - 

In [33]:
rnn_results, rnn_probabilities, rnn_predictions, rnn_cm = evaluate_model(
    model=rnn_model,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="RNN Baseline",
    batch_size=64
)

RNN Baseline RESULTS
Accuracy : 0.50534
Precision: 0.59000
Recall   : 0.04741
F1 Score : 0.08776
ROC-AUC  : 0.51143

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5018    0.9668    0.6607      2470
    Positive     0.5900    0.0474    0.0878      2489

    accuracy                         0.5053      4959
   macro avg     0.5459    0.5071    0.3742      4959
weighted avg     0.5461    0.5053    0.3731      4959

Confusion Matrix
[[2388   82]
 [2371  118]]


In [34]:
rnn_model.save("rnn_baseline.keras")
print("saved")

saved


In [35]:
with open("rnn_baseline_history.pkl", "wb") as file:
    pickle.dump(rnn_history.history, file)

print("history saved")

history saved


In [52]:
baseline_results.append(rnn_results)

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750151,0.967115,0.519888,0.676248,0.952254
1,RNN Baseline,0.505344,0.590000,0.047409,0.087765,0.511427


### 3. GRU

In [25]:
gru_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GRU(128),
    Dense(64, activation="relu"
    ),
    Dense(
        1,
        activation="sigmoid"
    )
])

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,947,393 (15.06 MB)

 Trainable params: 3,947,393 (15.06 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
gru_history = gru_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 39ms/step - accuracy: 0.5074 - loss: 0.6931 - val_accuracy: 0.5024 - val_loss: 0.6926
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.5189 - loss: 0.6831 - val_accuracy: 0.5077 - val_loss: 0.6975
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.5742 - loss: 0.6455 - val_accuracy: 0.7572 - val_loss: 0.5530
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.8807 - loss: 0.2998 - val_accuracy: 0.8883 - val_loss: 0.3019
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9575 - loss: 0.1258 - val_accuracy: 0.8917 - val_loss: 0.3053
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.9844 - loss: 0.0559 - val_accuracy: 0.8901 - val_loss: 0.3773
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.9928 - loss: 0.0297 - val_accuracy: 0.8901 - val_loss: 0.4478
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9953 - loss: 0.0188 - 

In [27]:
gru_results, gru_probabilities, gru_predictions, gru_cm = evaluate_model(
    model=gru_model,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Baseline",
    batch_size=64
)

GRU Baseline RESULTS
Accuracy : 0.88465
Precision: 0.88202
Recall   : 0.88911
F1 Score : 0.88555
ROC-AUC  : 0.94349

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8873    0.8802    0.8837      2470
    Positive     0.8820    0.8891    0.8856      2489

    accuracy                         0.8847      4959
   macro avg     0.8847    0.8846    0.8846      4959
weighted avg     0.8847    0.8847    0.8847      4959

Confusion Matrix
[[2174  296]
 [ 276 2213]]


In [53]:
baseline_results.append(gru_results)

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750151,0.967115,0.519888,0.676248,0.952254
1,RNN Baseline,0.505344,0.590000,0.047409,0.087765,0.511427
2,GRU Baseline,0.884654,0.882025,0.889112,0.885554,0.943492


In [29]:
gru_model.save("gru_baseline.keras")
print("saved")

saved


In [30]:
with open("gru_baseline_history.pkl", "wb") as file:
    pickle.dump(gru_history.history, file)

print("history saved")

history saved


### 4. LSTM

In [10]:
from tensorflow.keras import Sequential, Input
lstm_model = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(128),

    Dense(64, activation="relu"),

    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
lstm_history = lstm_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.5071 - loss: 0.6935 - val_accuracy: 0.5083 - val_loss: 0.6929
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.5204 - loss: 0.6827 - val_accuracy: 0.5161 - val_loss: 0.6893
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.5355 - loss: 0.6561 - val_accuracy: 0.5129 - val_loss: 0.7065
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5394 - loss: 0.6450 - val_accuracy: 0.5157 - val_loss: 0.7525
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.6200 - loss: 0.5930 - val_accuracy: 0.8344 - val_loss: 0.4072
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.8947 - loss: 0.2681 - val_accuracy: 0.8929 - val_loss: 0.2877
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9546 - loss: 0.1334 - val_accuracy: 0.8885 - val_loss: 0.3146
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.9815 - loss: 0.0671 - 

In [12]:
lstm_results, lstm_probabilities, lstm_predictions, lstm_cm = evaluate_model(
    model=lstm_model,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="LSTM Baseline",
    batch_size=64
)

LSTM Baseline RESULTS
Accuracy : 0.88264
Precision: 0.87913
Recall   : 0.88831
F1 Score : 0.88369
ROC-AUC  : 0.94016

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8863    0.8769    0.8816      2470
    Positive     0.8791    0.8883    0.8837      2489

    accuracy                         0.8826      4959
   macro avg     0.8827    0.8826    0.8826      4959
weighted avg     0.8827    0.8826    0.8826      4959

Confusion Matrix
[[2166  304]
 [ 278 2211]]


In [54]:
baseline_results.append(lstm_results)

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750151,0.967115,0.519888,0.676248,0.952254
1,RNN Baseline,0.505344,0.590000,0.047409,0.087765,0.511427
2,GRU Baseline,0.884654,0.882025,0.889112,0.885554,0.943492
3,LSTM Baseline,0.882638,0.879125,0.888309,0.883693,0.940162


In [16]:
lstm_model.save("lstm_baseline.keras")
print("Saved")


Saved


In [17]:
with open("lstm_baseline_history.pkl", "wb") as file:
    pickle.dump(lstm_history.history, file)

print("history saved")

history saved


### 5. Bi-LSTM

In [18]:
bilstm_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    Bidirectional(
        LSTM(128)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
bilstm_history = bilstm_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 39s 59ms/step - accuracy: 0.7231 - loss: 0.5495 - val_accuracy: 0.6956 - val_loss: 0.6147
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 52ms/step - accuracy: 0.7077 - loss: 0.5567 - val_accuracy: 0.5048 - val_loss: 0.7610
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.8651 - loss: 0.3207 - val_accuracy: 0.8957 - val_loss: 0.2663
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.9368 - loss: 0.1716 - val_accuracy: 0.8804 - val_loss: 0.3005
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9625 - loss: 0.1113 - val_accuracy: 0.8945 - val_loss: 0.3057
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 56ms/step - accuracy: 0.9788 - loss: 0.0699 - val_accuracy: 0.8887 - val_loss: 0.3539
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 39s 53ms/step - accuracy: 0.9882 - loss: 0.0439 - val_accuracy: 0.8903 - val_loss: 0.4537
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9914 - loss: 0.0326 - 

In [20]:
bilstm_test_loss, bilstm_test_accuracy = bilstm_model.evaluate(
    X_test_integer,
    y_test,
    batch_size=64,
    verbose=0
)

print("Bi-LSTM Test Loss     :", round(bilstm_test_loss, 5))
print("Bi-LSTM Test Accuracy :", round(bilstm_test_accuracy, 5))

Bi-LSTM Test Loss     : 0.56512
Bi-LSTM Test Accuracy : 0.8782


In [21]:
bilstm_results, bilstm_probabilities, bilstm_predictions, bilstm_cm = evaluate_model(
    model=bilstm_model,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="Bi-LSTM Baseline",
    batch_size=64
)

Bi-LSTM Baseline RESULTS
Accuracy : 0.87820
Precision: 0.89386
Recall   : 0.85938
F1 Score : 0.87628
ROC-AUC  : 0.94213

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8636    0.8972    0.8801      2470
    Positive     0.8939    0.8594    0.8763      2489

    accuracy                         0.8782      4959
   macro avg     0.8787    0.8783    0.8782      4959
weighted avg     0.8788    0.8782    0.8782      4959

Confusion Matrix
[[2216  254]
 [ 350 2139]]


In [22]:
bilstm_model.save("bilstm_baseline.keras")
print("saved")

saved


In [23]:
with open("bilstm_baseline_history.pkl","wb") as file:
    pickle.dump(bilstm_history.history, file)
print("History Savedd")

History Savedd


In [55]:
baseline_results.append(bilstm_results)

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,ANN Baseline,0.750151,0.967115,0.519888,0.676248,0.952254
1,RNN Baseline,0.505344,0.590000,0.047409,0.087765,0.511427
2,GRU Baseline,0.884654,0.882025,0.889112,0.885554,0.943492
3,LSTM Baseline,0.882638,0.879125,0.888309,0.883693,0.940162
4,Bi-LSTM Baseline,0.878201,0.893857,0.859381,0.876280,0.942127


In [57]:
baseline_results_df.to_csv("phase4_baseline_comparison.csv", index=False)
print("saved")

saved


In [58]:
print("ANN Test Accuracy:")
print(ann_results["Accuracy"])

print("\nActual distribution:")
print(np.bincount(y_test))

print("\nPredicted distribution:")
print(np.bincount(ann_predictions))

ANN Test Accuracy:
0.7501512401693889

Actual distribution:
[2470 2489]

Predicted distribution:
[3621 1338]


In [59]:
print("Bi-LSTM Test Accuracy:")
print(bilstm_results["Accuracy"])

print("\nActual distribution:")
print(np.bincount(y_test))

print("\nPredicted distribution:")
print(np.bincount(bilstm_predictions))

Bi-LSTM Test Accuracy:
0.878201250252067

Actual distribution:
[2470 2489]

Predicted distribution:
[2566 2393]


In [60]:
print("=" * 50)
print("Bi-LSTM TEST CHECK")
print("=" * 50)

print("Test Accuracy:", bilstm_results["Accuracy"])

print("\nActual distribution:")
print(np.bincount(y_test))

print("\nPredicted distribution:")
print(np.bincount(bilstm_predictions))

print("\nPrediction probabilities:")
print("Minimum:", bilstm_probabilities.min())
print("Maximum:", bilstm_probabilities.max())
print("Mean:", bilstm_probabilities.mean())

print("\nConfusion Matrix:")
print(bilstm_cm)

Bi-LSTM TEST CHECK
Test Accuracy: 0.878201250252067

Actual distribution:
[2470 2489]

Predicted distribution:
[2566 2393]

Prediction probabilities:
Minimum: 0.000101320584
Maximum: 0.9999652
Mean: 0.4824727

Confusion Matrix:
[[2216  254]
 [ 350 2139]]


In [61]:
models_check = [
    ("ANN", ann_results, ann_probabilities, ann_predictions, ann_cm),
    ("RNN", rnn_results, rnn_probabilities, rnn_predictions, rnn_cm),
    ("GRU", gru_results, gru_probabilities, gru_predictions, gru_cm),
    ("LSTM", lstm_results, lstm_probabilities, lstm_predictions, lstm_cm),
    ("Bi-LSTM", bilstm_results, bilstm_probabilities, bilstm_predictions, bilstm_cm)
]

for name, results, probabilities, predictions, cm in models_check:

    print("\n" + "=" * 60)
    print(f"{name} TEST CHECK")
    print("=" * 60)

    print("Test Accuracy:", results["Accuracy"])

    print("\nActual distribution:")
    print(np.bincount(y_test))

    print("\nPredicted distribution:")
    print(np.bincount(predictions))

    print("\nPrediction probabilities:")
    print("Minimum:", probabilities.min())
    print("Maximum:", probabilities.max())
    print("Mean:", probabilities.mean())

    print("\nConfusion Matrix:")
    print(cm)


ANN TEST CHECK
Test Accuracy: 0.7501512401693889

Actual distribution:
[2470 2489]

Predicted distribution:
[3621 1338]

Prediction probabilities:
Minimum: 4.7741948e-14
Maximum: 1.0
Mean: 0.29146847

Confusion Matrix:
[[2426   44]
 [1195 1294]]

RNN TEST CHECK
Test Accuracy: 0.505343819318411

Actual distribution:
[2470 2489]

Predicted distribution:
[4759  200]

Prediction probabilities:
Minimum: 0.058436025
Maximum: 0.993821
Mean: 0.48435453

Confusion Matrix:
[[2388   82]
 [2371  118]]

GRU TEST CHECK
Test Accuracy: 0.8846541641459972

Actual distribution:
[2470 2489]

Predicted distribution:
[2450 2509]

Prediction probabilities:
Minimum: 1.6977054e-08
Maximum: 1.0
Mean: 0.5066266

Confusion Matrix:
[[2174  296]
 [ 276 2213]]

LSTM TEST CHECK
Test Accuracy: 0.882637628554144

Actual distribution:
[2470 2489]

Predicted distribution:
[2444 2515]

Prediction probabilities:
Minimum: 2.438949e-07
Maximum: 1.0
Mean: 0.506025

Confusion Matrix:
[[2166  304]
 [ 278 2211]]

Bi-LSTM TEST 